# Predictive Statistical Analysis of Influencer Marketing ROI
## Using Multiple Linear Regression (Python + Statsmodels)

---

| | |
|---|---|
| **Author** | Neema Sree |
| **Project Type** | Final Year Academic Project — Statistical Analysis |
| **Dataset** | Publicly available Influencer Marketing ROI dataset (Kaggle, 150,000 rows) |
| **Raw Rows** | 150,000 |
| **Processed Rows** | 147,000 |
| **Regression Observations** | 147,000 (full processed dataset, no sampling) |


---
## Module 1 — Project Introduction


### Project Title
**Predictive Statistical Analysis of Influencer Marketing ROI**

### Problem Statement
Companies spend millions on influencer marketing campaigns without knowing in advance which campaigns will deliver good ROI. Most budget allocation decisions are still made based on follower count and intuition, leading to inefficient spend.

### Aim
To build a multiple linear regression model using statistical methods that can predict the ROI of an influencer marketing campaign based on measurable campaign characteristics.

### Objectives
1. Load, inspect, and clean a real-world influencer marketing dataset
2. Engineer the key financial variables (Engagement Rate, Campaign Cost, Revenue, ROI)
3. Perform exploratory data analysis and descriptive statistics
4. Analyse feature correlations and identify predictors of ROI
5. Build and interpret a Multiple Linear Regression model using Statsmodels OLS
6. Diagnose model assumptions and evaluate model performance
7. Generate ROI predictions for new campaigns with confidence intervals

### Technologies Used
| Technology | Purpose |
|---|---|
| Python 3 | Core programming language |
| Pandas | Data loading, cleaning, manipulation |
| NumPy | Numerical operations |
| Matplotlib / Seaborn | Data visualisation |
| Statsmodels | OLS Regression — coefficients, p-values, R2, CIs |
| SciPy | Normality testing (Shapiro-Wilk) |
| scikit-learn | MAE and RMSE evaluation only |
| openpyxl | Excel workbook generation |

> **No machine learning frameworks, web frameworks, or databases are used.**  
> This is a pure statistical analysis project.

### Dataset Source
The raw dataset is a **publicly available dataset** sourced from Kaggle.  
It was **not collected by the author**.  
The project's contribution is the **data cleaning pipeline, feature engineering, and statistical modelling** applied to derive the processed dataset and regression results.

> **Important:** `Campaign_Cost`, `Revenue`, and `ROI` are **derived variables** calculated by this project's feature engineering. They were **not** provided by the original Kaggle dataset.


---
## Module 2 — Import Libraries


In [ ]:
import os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
PALETTE = ['#4C72B0','#DD8452','#55A868','#C44E52','#8172B2','#937860','#DA8BC3']

print('Library versions:')
print(f'  pandas      {pd.__version__}')
print(f'  numpy       {np.__version__}')
print(f'  statsmodels {sm.__version__}')
print('All libraries loaded successfully.')


---
## Module 3 — Load Raw Dataset

The raw Kaggle dataset is loaded from `data/raw/` **without any modification**.
This preserves the original source file as-is.


In [ ]:
RAW_PATH  = '../data/raw/influencer_marketing_roi_dataset.csv'
PROC_PATH = '../data/processed/influencer_roi_analysis_dataset.csv'

raw = pd.read_csv(RAW_PATH)
print(f'Shape          : {raw.shape}')
print(f'Columns        : {list(raw.columns)}')
raw.head(5)


In [ ]:
print('Data Types:')
print(raw.dtypes)
print(f'\nMissing values per column:')
print(raw.isnull().sum())
print(f'\nDuplicate rows : {raw.duplicated().sum():,}')


In [ ]:
print('Unique values per categorical column:')
for col in ['platform','influencer_category','campaign_type']:
    print(f'  {col}: {sorted(raw[col].unique().tolist())}')
print()
print('Numeric summary:')
raw[['engagements','estimated_reach','product_sales','campaign_duration_days']].describe().round(2)


---
## Module 4 — Data Cleaning

### Cleaning Steps Applied
1. Strip whitespace from string columns
2. Parse `start_date` and `end_date` to datetime
3. Drop rows with missing values
4. Drop exact duplicate rows
5. Retain only rows with recognised platform and category values
6. Remove rows where `estimated_reach = 0` (would cause division by zero in Engagement Rate)
7. Remove rows with negative numeric values
8. Validate `campaign_duration_days` against computed date difference (tolerance ±2 days)


In [ ]:
# Lookup tables — same values used in feature engineering and prediction
CPM = {'Instagram':150, 'YouTube':120, 'TikTok':100, 'Twitter':80}
AVG_UNIT_VALUE = {
    'Beauty':1500, 'Tech':5000, 'Fashion':1200, 'Food':300,
    'Fitness':800, 'Travel':3000, 'Gaming':1000
}

df = raw.copy()
initial_rows = len(df)

# Step 1: Strip whitespace
for col in ['platform','influencer_category','campaign_type']:
    df[col] = df[col].str.strip()
print('Step 1: Whitespace stripped.')

# Step 2: Parse dates
df['start_date'] = pd.to_datetime(df['start_date'], errors='coerce')
df['end_date']   = pd.to_datetime(df['end_date'],   errors='coerce')
print('Step 2: Dates parsed.')

# Step 3: Drop missing values
before = len(df); df.dropna(inplace=True)
print(f'Step 3: Missing values removed : {before - len(df):,} rows')

# Step 4: Drop duplicates
before = len(df); df.drop_duplicates(inplace=True)
print(f'Step 4: Duplicates removed     : {before - len(df):,} rows')

# Step 5: Valid categories
before = len(df)
df = df[df['platform'].isin(CPM) & df['influencer_category'].isin(AVG_UNIT_VALUE)]
print(f'Step 5: Invalid values removed : {before - len(df):,} rows')

# Step 6: Zero reach
before = len(df); df = df[df['estimated_reach'] > 0]
print(f'Step 6: Zero-reach removed     : {before - len(df):,} rows')

# Step 7: Negative numerics
for col in ['engagements','product_sales','campaign_duration_days']:
    before = len(df); df = df[df[col] >= 0]
    if before - len(df): print(f'Step 7: Negative {col} removed: {before-len(df):,}')

# Step 8: Duration validation
before = len(df)
computed = (df['end_date'] - df['start_date']).dt.days
df = df[(computed - df['campaign_duration_days']).abs() <= 2]
print(f'Step 8: Duration mismatch removed: {before - len(df):,} rows')

print(f'\nCleaning Summary:')
print(f'  Original rows : {initial_rows:,}')
print(f'  Rows retained : {len(df):,}')
print(f'  Rows removed  : {initial_rows - len(df):,}')


---
## Module 5 — Feature Engineering

> **These four variables do not exist in the raw dataset.**  
> They are calculated by this project using documented formulas and assumptions.

| Variable | Formula | Notes |
|---|---|---|
| `Engagement_Rate` | `(Engagements / Estimated_Reach) × 100` | % — how effectively the audience engaged |
| `Campaign_Cost` | `(Estimated_Reach × Platform_CPM) / 1000` | INR — cost based on CPM model |
| `Revenue` | `Product_Sales × Category_Avg_Unit_Value` | INR — estimated revenue |
| `ROI` | `(Revenue − Campaign_Cost) / Campaign_Cost` | Target variable — dimensionless ratio |

**CPM rates (INR):** Instagram = 150 | YouTube = 120 | TikTok = 100 | Twitter = 80

**Average unit values (INR):** Beauty = 1,500 | Tech = 5,000 | Fashion = 1,200 | Food = 300 | Fitness = 800 | Travel = 3,000 | Gaming = 1,000


In [ ]:
# Engagement Rate (%)
df['engagement_rate'] = (df['engagements'] / df['estimated_reach']) * 100
print('Engagement_Rate computed.')

# Campaign Cost (INR) — CPM model
df['campaign_cost'] = df.apply(
    lambda r: (r['estimated_reach'] * CPM[r['platform']]) / 1000, axis=1)
print('Campaign_Cost computed.')

# Revenue (INR)
df['revenue'] = df.apply(
    lambda r: r['product_sales'] * AVG_UNIT_VALUE[r['influencer_category']], axis=1)
print('Revenue computed.')

# Remove zero-cost rows before ROI calculation
df = df[df['campaign_cost'] > 0]

# ROI (target variable)
df['roi'] = (df['revenue'] - df['campaign_cost']) / df['campaign_cost']
print('ROI computed.')

# Outlier capping: retain 1st-99th percentile
lo, hi = df['roi'].quantile(0.01), df['roi'].quantile(0.99)
before = len(df)
df = df[(df['roi'] >= lo) & (df['roi'] <= hi)]
print(f'ROI outliers removed (1st-99th pct): {before - len(df):,} rows')
print(f'ROI range: {df["roi"].min():.4f} to {df["roi"].max():.4f}')

# Temporal features
df['year']  = df['start_date'].dt.year
df['month'] = df['start_date'].dt.month
print('Year and Month extracted.')


---
## Module 6 — Final Processed Dataset

Select and rename final columns to produce the analytical dataset.  
This matches `data/processed/influencer_roi_analysis_dataset.csv` exactly.


In [ ]:
COLS = {
    'campaign_id':'Campaign_ID','platform':'Platform',
    'influencer_category':'Influencer_Category','campaign_type':'Campaign_Type',
    'start_date':'Start_Date','end_date':'End_Date',
    'campaign_duration_days':'Campaign_Duration_Days',
    'estimated_reach':'Estimated_Reach','engagements':'Engagements',
    'product_sales':'Product_Sales','engagement_rate':'Engagement_Rate',
    'campaign_cost':'Campaign_Cost','revenue':'Revenue',
    'roi':'ROI','year':'Year','month':'Month',
}
final = df[list(COLS.keys())].rename(columns=COLS).copy()
final['Engagement_Rate'] = final['Engagement_Rate'].round(4)
final['Campaign_Cost']   = final['Campaign_Cost'].round(2)
final['Revenue']         = final['Revenue'].round(2)
final['ROI']             = final['ROI'].round(4)
final.sort_values(['Start_Date','Campaign_ID'], inplace=True)
final.reset_index(drop=True, inplace=True)

print(f'Final dataset shape : {final.shape}')
print(f'Null values         : {final.isnull().sum().sum()}')
print(f'Columns             : {list(final.columns)}')
final.head(5)


In [ ]:
print('Data types of final dataset:')
print(final.dtypes)
print()
print('Platform distribution:')
print(final['Platform'].value_counts().to_string())
print()
print('Category distribution:')
print(final['Influencer_Category'].value_counts().to_string())


---
## Module 7 — Exploratory Data Analysis


### Chart 1 — ROI Distribution


In [ ]:
fig, axes = plt.subplots(1,2,figsize=(14,5))
fig.suptitle('ROI Distribution — 147,000 Campaigns',fontsize=14,fontweight='bold')
roi_clip = final['ROI'].clip(upper=final['ROI'].quantile(0.95))
axes[0].hist(roi_clip,bins=60,color=PALETTE[0],edgecolor='white')
axes[0].set(title='Histogram of ROI',xlabel='ROI',ylabel='Frequency')
axes[1].plot(sorted(roi_clip),np.linspace(0,1,len(roi_clip)),color=PALETTE[0])
axes[1].set(title='Cumulative Distribution of ROI',xlabel='ROI',ylabel='Cumulative Proportion')
plt.tight_layout(); plt.savefig('../outputs/figures/01_roi_distribution.png',dpi=150,bbox_inches='tight'); plt.show()


### Chart 2 — Engagement Rate Distribution


In [ ]:
fig, axes = plt.subplots(1,2,figsize=(14,5))
fig.suptitle('Engagement Rate Distribution',fontsize=14,fontweight='bold')
er = final['Engagement_Rate'].clip(upper=final['Engagement_Rate'].quantile(0.95))
axes[0].hist(er,bins=60,color=PALETTE[1],edgecolor='white')
axes[0].set(title='Histogram',xlabel='Engagement Rate (%)',ylabel='Frequency')
axes[1].boxplot(er,vert=False,patch_artist=True,boxprops=dict(facecolor=PALETTE[1],alpha=0.7))
axes[1].set(title='Boxplot',xlabel='Engagement Rate (%)')
plt.tight_layout(); plt.savefig('../outputs/figures/02_engagement_rate_distribution.png',dpi=150,bbox_inches='tight'); plt.show()


### Chart 3 — Campaign Cost Distribution


In [ ]:
fig, axes = plt.subplots(1,2,figsize=(14,5))
fig.suptitle('Campaign Cost Distribution',fontsize=14,fontweight='bold')
axes[0].hist(final['Campaign_Cost'],bins=60,color=PALETTE[2],edgecolor='white')
axes[0].set(title='Histogram',xlabel='Campaign Cost (INR)',ylabel='Frequency')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f'{x/1000:.0f}K'))
axes[1].boxplot(final['Campaign_Cost'],vert=False,patch_artist=True,boxprops=dict(facecolor=PALETTE[2],alpha=0.7))
axes[1].set(title='Boxplot',xlabel='Campaign Cost (INR)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f'{x/1000:.0f}K'))
plt.tight_layout(); plt.savefig('../outputs/figures/03_campaign_cost_distribution.png',dpi=150,bbox_inches='tight'); plt.show()


### Chart 4 — Product Sales Distribution


In [ ]:
fig, axes = plt.subplots(1,2,figsize=(14,5))
fig.suptitle('Product Sales Distribution',fontsize=14,fontweight='bold')
axes[0].hist(final['Product_Sales'],bins=60,color=PALETTE[3],edgecolor='white')
axes[0].set(title='Histogram',xlabel='Units Sold',ylabel='Frequency')
axes[1].boxplot(final['Product_Sales'],vert=False,patch_artist=True,boxprops=dict(facecolor=PALETTE[3],alpha=0.7))
axes[1].set(title='Boxplot',xlabel='Units Sold')
plt.tight_layout(); plt.savefig('../outputs/figures/04_product_sales_distribution.png',dpi=150,bbox_inches='tight'); plt.show()


### Chart 5 — Campaign Cost vs Revenue


In [ ]:
sample = final.sample(n=5000,random_state=42)
fig, ax = plt.subplots(figsize=(9,6))
ax.scatter(sample['Campaign_Cost'],sample['Revenue'],alpha=0.25,s=12,color=PALETTE[0])
ax.set(title='Campaign Cost vs Revenue',xlabel='Campaign Cost (INR)',ylabel='Revenue (INR)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f'{x/1000:.0f}K'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f'{x/1e6:.1f}M'))
plt.tight_layout(); plt.savefig('../outputs/figures/05_cost_vs_revenue.png',dpi=150,bbox_inches='tight'); plt.show()


### Chart 6 — Engagement Rate vs ROI


In [ ]:
sample = final.sample(n=5000,random_state=42)
fig, ax = plt.subplots(figsize=(9,6))
ax.scatter(sample['Engagement_Rate'].clip(upper=100),sample['ROI'].clip(upper=500),alpha=0.25,s=12,color=PALETTE[1])
ax.set(title='Engagement Rate vs ROI',xlabel='Engagement Rate (%)',ylabel='ROI')
plt.tight_layout(); plt.savefig('../outputs/figures/06_engagement_vs_roi.png',dpi=150,bbox_inches='tight'); plt.show()


### Charts 7-9 — ROI by Platform, Category, Campaign Type


In [ ]:
# ROI by Platform
fig, axes = plt.subplots(1,2,figsize=(14,5))
fig.suptitle('ROI by Platform',fontsize=14,fontweight='bold')
pc = final['Platform'].value_counts()
axes[0].bar(pc.index,pc.values,color=PALETTE[:4])
axes[0].set(title='Campaign Count by Platform',xlabel='Platform',ylabel='Count')
for i,v in enumerate(pc.values): axes[0].text(i,v+200,f'{v:,}',ha='center',fontsize=9)
final.boxplot(column='ROI',by='Platform',ax=axes[1],patch_artist=True,showfliers=False)
axes[1].set(title='ROI by Platform',xlabel='Platform',ylabel='ROI'); plt.suptitle('')
plt.tight_layout(); plt.savefig('../outputs/figures/07_roi_by_platform.png',dpi=150,bbox_inches='tight'); plt.show()


In [ ]:
# ROI by Category
fig, axes = plt.subplots(1,2,figsize=(14,5))
fig.suptitle('ROI by Influencer Category',fontsize=14,fontweight='bold')
cat = final.groupby('Influencer_Category')['ROI'].mean().sort_values(ascending=False)
axes[0].barh(cat.index,cat.values,color=PALETTE[:7])
axes[0].set(title='Average ROI by Category',xlabel='Mean ROI')
for i,v in enumerate(cat.values): axes[0].text(v+0.5,i,f'{v:.1f}',va='center',fontsize=9)
final.boxplot(column='ROI',by='Influencer_Category',ax=axes[1],patch_artist=True,showfliers=False,vert=False)
axes[1].set(title='ROI Spread by Category',xlabel='ROI'); plt.suptitle('')
plt.tight_layout(); plt.savefig('../outputs/figures/08_roi_by_category.png',dpi=150,bbox_inches='tight'); plt.show()


In [ ]:
# ROI by Campaign Type & Monthly Trend
fig, axes = plt.subplots(1,2,figsize=(14,5))
fig.suptitle('ROI by Campaign Type & Monthly Trend',fontsize=14,fontweight='bold')
ct = final.groupby('Campaign_Type')['ROI'].mean().sort_values(ascending=False)
axes[0].bar(ct.index,ct.values,color=PALETTE[:5])
axes[0].set(title='Average ROI by Campaign Type',xlabel='Campaign Type',ylabel='Mean ROI')
axes[0].tick_params(axis='x',rotation=15)
monthly = final.groupby('Month')['ROI'].mean()
axes[1].plot(monthly.index,monthly.values,marker='o',color=PALETTE[2])
axes[1].set(title='Average ROI by Month',xlabel='Month',ylabel='Mean ROI')
axes[1].set_xticks(range(1,13))
axes[1].set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
plt.tight_layout(); plt.savefig('../outputs/figures/09_roi_by_campaign_type.png',dpi=150,bbox_inches='tight'); plt.show()


---
## Module 8 — Descriptive Statistics


In [ ]:
NUM_COLS = ['Estimated_Reach','Engagements','Product_Sales',
            'Campaign_Duration_Days','Engagement_Rate',
            'Campaign_Cost','Revenue','ROI']

desc = final[NUM_COLS].describe().T
desc['Skewness'] = final[NUM_COLS].skew()
desc['Kurtosis'] = final[NUM_COLS].kurt()

# Add Mode
desc['Mode'] = [round(float(final[c].mode().iloc[0]),4) for c in NUM_COLS]

print('DESCRIPTIVE STATISTICS')
print('=' * 80)
print(desc[['count','mean','50%','Mode','std','min','max','Skewness','Kurtosis']]
      .rename(columns={'50%':'Median','std':'Std Dev','mean':'Mean','count':'Count'})
      .round(4).to_string())


In [ ]:
# Shapiro-Wilk Normality Test on ROI
# Full dataset too large; n=5,000 sample is standard practice for this test
sample_sw = final['ROI'].sample(n=5000, random_state=42)
stat, p = stats.shapiro(sample_sw)
print(f'Shapiro-Wilk Test for ROI Normality (sample n=5,000):')
print(f'  W statistic : {stat:.4f}')
print(f'  p-value     : {p:.6f}')
print(f'  Conclusion  : {"NOT normally distributed" if p < 0.05 else "Normally distributed"} (alpha=0.05)')
print()
print('Note: Non-normality of ROI is expected for large real-world datasets.')
print('OLS regression remains valid due to the large sample size (Central Limit Theorem).')


---
## Module 9 — Correlation Analysis

> **Correlation vs Causation:** A high Pearson correlation between two variables means they move together linearly. It does **not** prove that one variable causes changes in the other. Interpretation must remain statistical.


In [ ]:
corr = final[NUM_COLS].corr()
print('Pearson Correlation Matrix:')
print(corr.round(4).to_string())


In [ ]:
# Correlation Heatmap
fig, ax = plt.subplots(figsize=(11,9))
mask = np.triu(np.ones_like(corr,dtype=bool))
sns.heatmap(corr,mask=mask,annot=True,fmt='.2f',cmap='coolwarm',
            center=0,linewidths=0.5,ax=ax)
ax.set_title('Pearson Correlation Heatmap',fontsize=14,fontweight='bold')
plt.tight_layout(); plt.savefig('../outputs/figures/10_correlation_heatmap.png',dpi=150,bbox_inches='tight'); plt.show()


In [ ]:
# ROI Correlation Bar Chart
roi_corr = corr['ROI'].drop('ROI').sort_values(ascending=False)
colors = [PALETTE[0] if v>0 else PALETTE[3] for v in roi_corr.values]
fig, ax = plt.subplots(figsize=(9,5))
ax.barh(roi_corr.index,roi_corr.values,color=colors)
ax.axvline(0,color='black',linewidth=0.8)
ax.set(title='Feature Correlation with ROI',xlabel='Pearson r')
for i,v in enumerate(roi_corr.values):
    ax.text(v+0.002 if v>=0 else v-0.002,i,f'{v:.3f}',va='center',
            ha='left' if v>=0 else 'right',fontsize=9)
plt.tight_layout(); plt.savefig('../outputs/figures/11_roi_correlation_bar.png',dpi=150,bbox_inches='tight'); plt.show()
print('\nCorrelation with ROI (ranked):')
print(roi_corr.round(4).to_string())


---
## Module 10 — Multiple Linear Regression (OLS)

**Method:** Ordinary Least Squares (OLS) via `statsmodels.api.OLS`  
**Observations:** All **147,000** processed rows — no random sampling  
**Dependent variable (Target):** `ROI`  
**Independent variables (Predictors):** `Engagement_Rate`, `Campaign_Cost`, `Product_Sales`, `Campaign_Duration_Days`

A constant term is added using `sm.add_constant()` to estimate the intercept.


In [ ]:
FEATURES = ['Engagement_Rate','Campaign_Cost','Product_Sales','Campaign_Duration_Days']
TARGET   = 'ROI'

X = final[FEATURES]
y = final[TARGET]
X_const = sm.add_constant(X)

print(f'Fitting OLS on full processed dataset: n = {len(final):,} observations')
print(f'Predictors: {FEATURES}')
print(f'Target    : {TARGET}')
model = sm.OLS(y, X_const).fit()
print(model.summary())


In [ ]:
# Print coefficients clearly
coefs  = model.params
pvals  = model.pvalues
conf   = model.conf_int()

print('REGRESSION COEFFICIENTS')
print('=' * 65)
print(f'{"Variable":<30} {"Coef":>12} {"p-value":>12} {"Sig":>6}')
print('-' * 65)
for feat in ['const'] + FEATURES:
    sig = '***' if pvals[feat]<0.001 else ('**' if pvals[feat]<0.01 else ('*' if pvals[feat]<0.05 else 'ns'))
    print(f'{feat:<30} {coefs[feat]:>12.6f} {pvals[feat]:>12.6f} {sig:>6}')
print('=' * 65)
print('Significance: *** p<0.001  ** p<0.01  * p<0.05  ns: not significant')


---
## Module 11 — Regression Interpretation

### Final Regression Equation

```
ROI = 161.8977
    + 1.0914  x Engagement_Rate
    - 0.00286 x Campaign_Cost
    + 0.0567  x Product_Sales
    - 0.0088  x Campaign_Duration_Days
```

### Coefficient Interpretation

| Predictor | Coefficient | Interpretation |
|---|---|---|
| Engagement_Rate | +1.0914 | For every 1% increase in engagement rate, predicted ROI increases by 1.09 units, holding all other predictors constant. **Strongest positive predictor.** |
| Campaign_Cost | -0.00286 | For every INR 1 increase in campaign cost, predicted ROI decreases by 0.00286 units. Higher spending on reach alone does not guarantee better returns. |
| Product_Sales | +0.0567 | For every additional unit sold, predicted ROI increases by 0.057. Campaigns that drive actual purchases deliver better ROI. |
| Campaign_Duration_Days | -0.0088 | p = 0.916. **Not statistically significant.** Duration alone does not predict ROI. |

> **Important:** These relationships are **statistical associations**, not causal claims.  
> The model does not prove that changing engagement rate *causes* ROI to change.

### R-squared Interpretation
R2 = 0.3190 means the four predictors together explain approximately **32% of the variance in ROI**.  
The remaining 68% is explained by factors not captured in this dataset  
(e.g. creative quality, brand strength, seasonality, competitor activity).


In [ ]:
# Coefficient Plot with 95% Confidence Intervals
features_plot = FEATURES
fig, ax = plt.subplots(figsize=(9,5))
for i, f in enumerate(features_plot):
    ax.plot([conf.loc[f,0],conf.loc[f,1]],[i,i],color=PALETTE[0],linewidth=2)
    color = PALETTE[0] if coefs[f]>0 else PALETTE[3]
    ax.scatter(coefs[f],i,color=color,s=80,zorder=5)
    ax.text(coefs[f],i+0.15,f'{coefs[f]:.4f}',ha='center',fontsize=9)
ax.axvline(0,color='black',linewidth=0.8,linestyle='--')
ax.set_yticks(range(len(features_plot))); ax.set_yticklabels(features_plot)
ax.set(title='OLS Coefficients with 95% Confidence Intervals',xlabel='Coefficient Value')
plt.tight_layout(); plt.savefig('../outputs/figures/12_coefficients.png',dpi=150,bbox_inches='tight'); plt.show()


---
## Module 12 — Model Diagnostics

OLS regression relies on several assumptions. These plots help assess them:

| Diagnostic | What it checks |
|---|---|
| Q-Q Plot | Are residuals approximately normally distributed? |
| Residuals vs Fitted | Is there a pattern (heteroscedasticity)? |
| Actual vs Predicted | How well does the model fit? |
| Residual Distribution | Is the residual mean close to zero? |


In [ ]:
y_pred    = model.fittedvalues
residuals = model.resid

fig, axes = plt.subplots(1,3,figsize=(18,5))
fig.suptitle('Residual Analysis',fontsize=14,fontweight='bold')

# Q-Q Plot
(osm,osr),(slope,intercept,_) = stats.probplot(residuals,dist='norm')
axes[0].plot(osm,osr,'o',alpha=0.2,ms=3,color=PALETTE[0])
axes[0].plot(osm,slope*np.array(osm)+intercept,color=PALETTE[3],linewidth=1.5)
axes[0].set(title='Q-Q Plot\n(Deviation from line = non-normality)',
            xlabel='Theoretical Quantiles',ylabel='Sample Quantiles')

# Residuals vs Fitted
axes[1].scatter(y_pred,residuals,alpha=0.1,s=6,color=PALETTE[0])
axes[1].axhline(0,color=PALETTE[3],linewidth=1.5,linestyle='--')
axes[1].set(title='Residuals vs Fitted\n(Fan shape = heteroscedasticity)',
            xlabel='Fitted Values',ylabel='Residuals')

# Residual histogram
axes[2].hist(residuals,bins=60,color=PALETTE[0],edgecolor='white')
axes[2].set(title='Residual Distribution\n(Should be centred on 0)',
            xlabel='Residual',ylabel='Frequency')

plt.tight_layout(); plt.savefig('../outputs/figures/13_residual_analysis.png',dpi=150,bbox_inches='tight'); plt.show()
print(f'Durbin-Watson statistic: {sm.stats.stattools.durbin_watson(residuals):.4f}')
print('(Value near 2.0 = no autocorrelation in residuals)')


In [ ]:
# Actual vs Predicted
fig, ax = plt.subplots(figsize=(8,6))
ax.scatter(y,y_pred,alpha=0.1,s=6,color=PALETTE[0])
lims = [min(float(y.min()),float(y_pred.min())), max(float(y.max()),float(y_pred.max()))]
ax.plot(lims,lims,color=PALETTE[3],linewidth=1.5,linestyle='--',label='Perfect fit line')
ax.set(title='Actual vs Predicted ROI',xlabel='Actual ROI',ylabel='Predicted ROI')
ax.legend()
plt.tight_layout(); plt.savefig('../outputs/figures/14_actual_vs_predicted.png',dpi=150,bbox_inches='tight'); plt.show()


---
## Module 13 — Model Evaluation


In [ ]:
mae  = mean_absolute_error(y, y_pred)
rmse = float(np.sqrt(mean_squared_error(y, y_pred)))
r2   = model.rsquared
adj_r2 = model.rsquared_adj

print('MODEL EVALUATION SUMMARY  (n = 147,000)')
print('=' * 50)
print(f'  R-squared (R2)  : {r2:.4f}')
print(f'  Adjusted R2     : {adj_r2:.4f}')
print(f'  F-statistic     : {model.fvalue:.2f}  (p < 0.001)')
print(f'  AIC             : {model.aic:.2f}')
print(f'  BIC             : {model.bic:.2f}')
print(f'  MAE             : {mae:.4f}')
print(f'  RMSE            : {rmse:.4f}')
print(f'  Observations    : {int(model.nobs):,}')
print('=' * 50)


In [ ]:
errors = y - y_pred
fig, axes = plt.subplots(1,2,figsize=(14,5))
fig.suptitle('Model Evaluation',fontsize=14,fontweight='bold')
axes[0].hist(errors,bins=60,color=PALETTE[0],edgecolor='white')
axes[0].axvline(0,color=PALETTE[3],linewidth=1.5,linestyle='--')
axes[0].set(title=f'Error Distribution  (MAE={mae:.2f}, RMSE={rmse:.2f})',
            xlabel='Actual - Predicted',ylabel='Frequency')
axes[1].bar(['MAE','RMSE'],[mae,rmse],color=PALETTE[:2])
axes[1].set(title='Error Metrics',ylabel='Value')
for i,(label,val) in enumerate(zip(['MAE','RMSE'],[mae,rmse])):
    axes[1].text(i,val+1,f'{val:.2f}',ha='center',fontsize=11,fontweight='bold')
plt.tight_layout(); plt.savefig('../outputs/figures/15_error_distribution.png',dpi=150,bbox_inches='tight'); plt.show()


---
## Module 14 — ROI Prediction for New Campaigns

Predictions use the same feature engineering as training.  
Each prediction includes a **95% prediction interval** — the range within which a single new campaign's actual ROI is expected to fall.

> The wide prediction intervals reflect high variance in ROI across the dataset (RMSE = 267.18).  
> The point estimates are the model's best predictions given the inputs.


In [ ]:
new_campaigns = pd.DataFrame({
    'Campaign':['Micro-Influencer Instagram','Macro-Influencer YouTube',
                'TikTok Viral Campaign','Twitter Tech Launch','Long-Duration Fitness'],
    'Engagement_Rate':[8.5,2.1,15.3,4.2,6.8],
    'Campaign_Cost':[30000,90000,50000,25000,45000],
    'Product_Sales':[500,1200,800,300,700],
    'Campaign_Duration_Days':[14,30,7,10,60],
})

X_new = new_campaigns[FEATURES]
X_new_c = sm.add_constant(X_new,has_constant='add')
pred = model.get_prediction(X_new_c).summary_frame(alpha=0.05)

new_campaigns['Predicted_ROI'] = pred['mean'].values.round(4)
new_campaigns['CI_Lower_95']   = pred['obs_ci_lower'].values.round(4)
new_campaigns['CI_Upper_95']   = pred['obs_ci_upper'].values.round(4)

print(new_campaigns[['Campaign','Predicted_ROI','CI_Lower_95','CI_Upper_95']].to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(11,5))
bars = ax.bar(new_campaigns['Campaign'],new_campaigns['Predicted_ROI'],color=PALETTE[:5],zorder=3)
ax.errorbar(
    new_campaigns['Campaign'],new_campaigns['Predicted_ROI'],
    yerr=[new_campaigns['Predicted_ROI']-new_campaigns['CI_Lower_95'],
          new_campaigns['CI_Upper_95']-new_campaigns['Predicted_ROI']],
    fmt='none',color='black',capsize=5,linewidth=1.5,zorder=4
)
ax.set(title='Predicted ROI for New Campaigns (95% Prediction Interval)',ylabel='Predicted ROI')
ax.tick_params(axis='x',rotation=18)
for bar,val in zip(bars,new_campaigns['Predicted_ROI']):
    ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.5,f'{val:.1f}',ha='center',fontsize=9)
ax.grid(axis='y',alpha=0.4)
plt.tight_layout(); plt.savefig('../outputs/figures/16_predictions.png',dpi=150,bbox_inches='tight'); plt.show()


---
## Conclusion

### Final Regression Equation
```
ROI = 161.8977
    + 1.0914  x Engagement_Rate
    - 0.00286 x Campaign_Cost
    + 0.0567  x Product_Sales
    - 0.0088  x Campaign_Duration_Days
```

### Key Findings
1. **Engagement Rate** is the strongest positive predictor of ROI (coef = +1.09, p < 0.001)
2. **Product Sales** has a significant positive effect (coef = +0.057, p < 0.001)
3. **Campaign Cost** has a significant negative effect (coef = -0.00286, p < 0.001) — higher spending on reach does not guarantee higher ROI
4. **Campaign Duration** is not statistically significant (p = 0.916)

### Business Recommendations
- Prioritise **engagement rate** over follower count when selecting influencers
- Focus budgets on campaigns that **drive actual product purchases**
- Avoid scaling reach arbitrarily — **cost efficiency matters more than scale**
- Campaign duration alone does not improve ROI

### Limitations
- R2 = 0.319 — the model explains ~32% of ROI variance; other factors matter
- CPM and unit values are estimated assumptions, not actual invoice data
- Categorical variables not included in regression (used in EDA only)
- ROI is right-skewed; OLS is robust due to large n but non-normality noted
